Base → GRPO: полная validation двух сохранённых LR-пилотов

Обучение НЕ выполняется. Последовательно восстанавливаем merged-модели из адаптеров и оцениваем на 956 вопросах. Тестовые выборки не используются.

Перед запуском: GPU T4, Internet, прежний датасет Spider/BIRD и датасет с вашим архивом base_grpo_lr_complete_….zip через Add Input. Можно загрузить и распакованное содержимое этого архива. Не используйте архив base_grpo_reports_: там нет весов адаптеров. Secret COMET_API_KEY должен быть доступен notebook.

Выполните шесть ячеек по порядку. После установки пакетов, если torch уже импортировался, перезапустите kernel и продолжайте со второй ячейки. Обновление cuDNN стоит последним; не повторяйте общий pip install после него.

In [ ]:
%pip install "vllm==0.31.0" "transformers==5.17.0" "peft==0.21.2" "accelerate==1.15.0" "comet-ml==3.59.0" python-dotenv
%pip install --no-deps --index-url https://download.pytorch.org/whl/cu130 "torchaudio==2.11.0+cu130"
%pip uninstall -y torchao torchcodec
%pip install --no-deps --no-cache-dir --index-url https://download.pytorch.org/whl "nvidia-cudnn-cu13==9.23.0.39"

In [ ]:
import os
import sys
import json
import shutil
import subprocess
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from datetime import datetime, timezone

WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")
REPO = WORK / "text2sql-grpo-sft"
RUN = WORK / "text2sql_runs" / "v1"
BASE = "Qwen/Qwen3.5-2B"
REVISION = "15852e8c16360a2fea060d615a32b45270f8a8fc"

if (REPO / ".git").exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/dancravik/text2sql-grpo.git", str(REPO)], check=True)

DATA_DIR = None  # При необходимости укажите Path("/kaggle/input/.../...")
if DATA_DIR is None:
    roots = sorted({
        p.parents[1] for p in INPUT.rglob("train_spider.json")
        if p.parent.name == "spider"
        and (p.parents[1] / "minidev/MINIDEV/mini_dev_sqlite.json").is_file()
    })
    if len(roots) == 1:
        DATA_DIR = roots[0]
    elif roots:
        raise ValueError(f"Укажите DATA_DIR вручную: {roots}")
    else:
        DATA_DIR = WORK / "text2sql_data"
        if not (DATA_DIR / "spider/train_spider.json").is_file():
            archives = list(INPUT.rglob("text2sql-data.zip"))
            if len(archives) != 1:
                raise ValueError(f"Добавьте прежний датасет или задайте DATA_DIR: {archives}")
            with ZipFile(archives[0]) as archive:
                archive.extractall(DATA_DIR)

RUN.mkdir(parents=True, exist_ok=True)
os.environ["DATA_DIR"] = str(DATA_DIR)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ.pop("TORCH_CUDNN_V8_API_DISABLED", None)
subprocess.run([sys.executable, "scripts/freeze_splits.py", "--verify"], cwd=REPO, check=True)
print("DATA_DIR:", DATA_DIR)
print("Свободно GiB:", round(shutil.disk_usage(WORK).free / 1024**3, 2))

In [ ]:
from kaggle_secrets import UserSecretsClient

os.environ["COMET_API_KEY"] = UserSecretsClient().get_secret("COMET_API_KEY")
os.environ["COMET_WORKSPACE"] = "danchenko-rtm"
os.environ["COMET_PROJECT_NAME"] = "text2sql-grpo"

subprocess.run([sys.executable, "-u", "-c", r"""
import importlib.metadata
import torch
from transformers import Trainer
from peft import LoraConfig
from vllm import LLM
for package in ("torch", "transformers", "peft", "accelerate", "vllm", "comet-ml", "nvidia-cudnn-cu13"):
    print(package, importlib.metadata.version(package))
assert torch.cuda.device_count() == 1
x = torch.randn(1, 6144, 512, device="cuda", dtype=torch.float16, requires_grad=True)
w = torch.randn(6144, 1, 4, device="cuda", dtype=torch.float16)
torch.nn.functional.conv1d(x, w, groups=6144, padding=3).float().mean().backward()
torch.cuda.synchronize()
print("GPU и FP16 conv1d forward/backward OK")
from comet_tracking import start_experiment
experiment = start_experiment(name="sft_full_environment_check")
try:
    experiment.log_metric("environment_ready", 1)
finally:
    experiment.end()
"""], cwd=REPO, env={**os.environ, "PYTHONPATH": str(REPO / "src")}, check=True)

Восстановление адаптеров из Add Input. Архивы не скачиваются заново. Автопоиск допускает ровно одну копию каждого адаптера; при неоднозначности задайте ADAPTER_INPUT вручную.

In [ ]:
ADAPTER_INPUT = None  # Например Path("/kaggle/input/datasets/your-name/grpo-models")
search_root = Path(ADAPTER_INPUT) if ADAPTER_INPUT is not None else INPUT
stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
BUNDLE = RUN / f"base_grpo_full_validation_{stamp}"
BUNDLE.mkdir(parents=True, exist_ok=False)
labels = ("lr_5e-06", "lr_1e-05")

def find_training(root, label):
    return sorted({
        p.parent for p in root.rglob("run.json")
        if p.parent.name == "training"
        and p.parent.parent.name == label
        and (p.parent / "adapter/adapter_model.safetensors").is_file()
    })

# Поддерживаются и архив, и уже распакованный Kaggle dataset.
found = {label: find_training(search_root, label) for label in labels}
if not all(found.values()):
    archives = sorted(search_root.rglob("base_grpo_lr_complete_*.zip"))
    if len(archives) != 1:
        raise ValueError(
            "Добавьте архив base_grpo_lr_complete_*.zip через Add Input "
            f"или укажите ADAPTER_INPUT. Найдено архивов: {archives}"
        )
    extracted = WORK / f"restored_grpo_{stamp}"
    extracted.mkdir()
    with ZipFile(archives[0]) as z:
        # Принимаем только пути внутри каталога распаковки.
        for member in z.infolist():
            destination = (extracted / member.filename).resolve()
            if not destination.is_relative_to(extracted.resolve()):
                raise ValueError(f"Недопустимый путь в ZIP: {member.filename}")
        z.extractall(extracted)
    found = {label: find_training(extracted, label) for label in labels}

SOURCES = {}
for label in labels:
    if len(found[label]) != 1:
        raise ValueError(f"Для {label} ожидается один адаптер, найдены: {found[label]}")
    source = BUNDLE / "sources" / label
    source.mkdir(parents=True)
    shutil.copytree(found[label][0] / "adapter", source / "adapter")
    report = json.loads((found[label][0] / "run.json").read_text())
    assert report["status"] == "ok", report["status"]
    assert report["steps_done"] == 128, report["steps_done"]
    assert report["optimizer_steps_skipped"] == 0
    if "global_step" not in report:
        report["global_step"] = report["steps_done"]
    (source / "run.json").write_text(json.dumps(report, indent=2) + "\n")
    SOURCES[label] = source
    print(label, "готов; Comet experiment:", report["comet_experiment_key"])

print("Результаты сохраняются в:", BUNDLE)

Оценка двух моделей по очереди. Для каждой: экспорт на CPU, оценка на GPU 0, архив с адаптером и результатами в исходный Comet experiment. Инициализация vLLM может долго не показывать прогресс генерации. Строка Process alive означает только, что процесс ещё не завершился. После подтверждённого архива удаляется только созданная merged-модель. Если оценка прервалась, не повторяйте все ячейки: файлы и адаптеры остаются в BUNDLE.

In [ ]:
def run_logged(script, arguments, logfile, env):
    with logfile.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(
            [sys.executable, "-u", str(REPO / "scripts" / script), *map(str, arguments)],
            cwd=REPO, env=env, stdout=log, stderr=subprocess.STDOUT,
        )
        while True:
            try:
                status = process.wait(timeout=30)
                break
            except subprocess.TimeoutExpired:
                lines = logfile.read_text(errors="replace").splitlines()
                print(f"Process alive [{logfile.name}]: {lines[-1][-200:] if lines else 'starting'}", flush=True)
    if status != 0:
        print("\n".join(logfile.read_text(errors="replace").splitlines()[-25:]))
        raise RuntimeError(f"Ошибка {script}, код {status}. Лог: {logfile}")

UPLOAD_CODE = r"""
import sys,json
from pathlib import Path
from comet_tracking import start_experiment
source, archive = map(Path, sys.argv[1:])
report = json.loads((source / "run.json").read_text())
experiment = start_experiment(key=report["comet_experiment_key"])
try:
    experiment.log_asset(str(archive), file_name=archive.name, overwrite=True)
    if not experiment.flush(timeout=300):
        raise RuntimeError("Comet не подтвердил доставку. Локальный архив сохранён.")
finally:
    experiment.end()
"""

results = []
for label, source in SOURCES.items():
    exported = BUNDLE / "exports" / label
    exported.parent.mkdir(exist_ok=True)
    evaluation = BUNDLE / "evaluations" / label
    logs = BUNDLE / "logs"
    logs.mkdir(exist_ok=True)
    metrics_path = evaluation / "metrics.json"

    if not metrics_path.exists():
        if not (exported / "run.json").is_file():
            if exported.exists():
                raise RuntimeError(f"Есть незавершённый экспорт: {exported}; сначала проверьте лог.")
            if shutil.disk_usage(WORK).free < 6 * 1024**3:
                raise RuntimeError("Нужно минимум 6 GiB свободного диска для экспорта.")
            run_logged("export_sft.py", ["--run-dir", source, "--out-dir", exported],
                       logs / f"{label}_export.log", {**os.environ, "CUDA_VISIBLE_DEVICES": ""})
        run_logged("evaluate.py", [
            "--model", exported / "merged", "--name", label,
            "--datasets", "validation", "--out-dir", evaluation.parent,
            "--comet-run", exported / "run.json",
            "--n", 1, "--temperature", 0, "--enable-thinking", "off",
            "--max-tokens", 1024, "--max-model-len", 4096, "--seed", 42,
        ], logs / f"{label}_validation.log", {**os.environ, "CUDA_VISIBLE_DEVICES": "0"})
    else:
        print(label, "используем готовую оценку; повторяем архивацию.")

    evaluation_report = json.loads(metrics_path.read_text())
    metrics = evaluation_report["datasets"]["validation"]
    assert metrics["num_questions"] == 956
    training_report = json.loads((source / "run.json").read_text())
    assert evaluation_report["manifest_sha256"] == training_report["manifest_sha256"]

    archive = BUNDLE / f"{BUNDLE.name}_{label}.zip"
    with ZipFile(archive, "w", ZIP_DEFLATED, allowZip64=True) as z:
        z.write(source / "run.json", "run.json")
        for path in (source / "adapter").rglob("*"):
            if path.is_file():
                z.write(path, "adapter/" + path.relative_to(source / "adapter").as_posix())
        for path in evaluation.glob("*.json*"):
            z.write(path, "evaluation/" + path.name)
        for path in logs.glob(f"{label}_*.log"):
            z.write(path, "logs/" + path.name)
        z.write(REPO / "splits/v1.json", "splits/v1.json")

    subprocess.run([sys.executable, "-u", "-c", UPLOAD_CODE, str(source), str(archive)],
                   env={**os.environ, "PYTHONPATH": str(REPO / "src")}, check=True)
    if (exported / "merged").exists():
        shutil.rmtree(exported / "merged")
    results.append({"label": label, "learning_rate": training_report["args"]["lr"], "metrics": metrics})
    (BUNDLE / "full_validation_selection.json").write_text(json.dumps(results, indent=2) + "\n")
    print(label, f"Accuracy: {metrics['greedy_execution_accuracy']:.2%}", flush=True)

print("\nИтог:")
for result in sorted(results, key=lambda row: row["metrics"]["greedy_execution_accuracy"], reverse=True):
    print(result["label"], f"{result['metrics']['greedy_execution_accuracy']:.2%}")

Общий архив для скачивания: исходные адаптеры, отчёты, предсказания и логи обоих вариантов, код и версии окружения. Существующие вложенные ZIP и merged-модели не дублируются. Этот общий архив создаётся локально; отдельные архивы каждого LR предыдущая ячейка отправляет в Comet.

In [ ]:
ARCHIVE = WORK / f"{BUNDLE.name}_complete.zip"
with ZipFile(ARCHIVE, "w", ZIP_DEFLATED, allowZip64=True) as z:
    for path in sorted(BUNDLE.rglob("*")):
        if (path.is_file() and path.suffix != ".zip"
                and "merged" not in path.relative_to(BUNDLE).parts):
            z.write(path, BUNDLE.name + "/" + path.relative_to(BUNDLE).as_posix())
    for folder in ("src", "scripts"):
        for path in sorted((REPO / folder).rglob("*.py")):
            z.write(path, "repository/" + path.relative_to(REPO).as_posix())
    z.write(REPO / "splits/v1.json", "repository/splits/v1.json")
    z.writestr("environment/pip_freeze.txt", subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))
with ZipFile(ARCHIVE) as z:
    bad = z.testzip()
    if bad:
        raise RuntimeError(f"Повреждён файл: {bad}")
from IPython.display import FileLink, display
print("Скачайте через Files:", ARCHIVE)
print(f"Размер: {ARCHIVE.stat().st_size / 1024**2:.1f} MiB")
display(FileLink(ARCHIVE.relative_to(WORK).as_posix()))